# Generate fixed rate langmuir dataset

In [2]:
from __future__ import annotations
import argparse
import json
from pathlib import Path
import numpy as np
import pandas as pd


In [3]:
PACKAGE = Path.cwd().parent

ROOT = PROJECT = PROJECT_ROOT = PACKAGE / 'work'

SETTINGS = argparse.Namespace(out_dir=PACKAGE / 'work/binding', n_records=10000, concentration_min_m=3.1622776601683794e-11, concentration_max_m=1e-08, random_seed=7, time_stop_s=7200, time_step_s=60, kon=55800.0, koff=1.79e-05, protein_diameter_nm=6.52, lx_nm=421.5, ly_nm=345.63, height_nm=100.0, ellipse_width_nm=236.04, ellipse_length_nm=84.3, ellipse_centre_spacing_nm=185.5, ellipse_tilt_deg=69.5, spacing_factor=1.0)


In [4]:
def count_binding_sites(lx: float, ly: float, protein_diameter: float, spacing_factor: float, height_nm: float, ellipse_width_nm: float, ellipse_length_nm: float, ellipse_centre_spacing_nm: float, ellipse_tilt_deg: float) -> tuple[int, dict[str, int]]:
    radius = protein_diameter / 2.0
    spacing = spacing_factor * protein_diameter
    cx = lx / 2.0
    cy = ly / 2.0
    width_nm = ellipse_width_nm
    length_nm = ellipse_length_nm
    centre_spacing_nm = ellipse_centre_spacing_nm
    tilt_deg = ellipse_tilt_deg
    a = width_nm / 2.0
    b = length_nm / 2.0
    
    ellipses = [{'cx': cx - centre_spacing_nm / 2.0, 
                 'cy': cy, 
                 'a': a, 
                 'b': b, 
                 'angle': np.deg2rad(tilt_deg)}, 
                 {'cx': cx + centre_spacing_nm / 2.0, 
                  'cy': cy, 
                  'a': a, 
                  'b': b, 
                  'angle': np.deg2rad(-tilt_deg)}]
    
    x_vals = np.arange(spacing / 2.0, lx - spacing / 2.0 + 1e-09, spacing)
    y_vals = np.arange(spacing / 2.0, ly - spacing / 2.0 + 1e-09, spacing)
    x_grid, y_grid = np.meshgrid(x_vals, y_vals)
    inside = is_double_ellipse_material(x_grid.ravel(), y_grid.ravel(), ellipses)
    n_plane = int((~inside).sum())
    n_top = int(inside.sum())
    nz = max(1, round(height_nm / spacing))
    n_side = 0

    for idx, ellipse in enumerate(ellipses):
        nt = round(ellipse_perimeter(ellipse['a'], ellipse['b']) / spacing)
        for theta in np.linspace(0.0, 2.0 * np.pi, nt + 1)[:-1]:
            ellipse = ellipse
            theta = theta
            ca = np.cos(ellipse['angle'])
            sa = np.sin(ellipse['angle'])
            x_local = ellipse['a'] * np.cos(theta)
            y_local = ellipse['b'] * np.sin(theta)
            normal_local = np.array([np.cos(theta) / ellipse['a'], np.sin(theta) / ellipse['b']])
            normal_local /= np.linalg.norm(normal_local)
            x = ellipse['cx'] + ca * x_local - sa * y_local
            y = ellipse['cy'] + sa * x_local + ca * y_local
            
            nx = ca * normal_local[0] - sa * normal_local[1]
            ny = sa * normal_local[0] + ca * normal_local[1]
            x0, y0, nx, ny = (float(x), float(y), float(nx), float(ny))
            xs = x0 + radius * nx
            ys = y0 + radius * ny
            if xs < 0.0 or xs > lx or ys < 0.0 or (ys > ly):
                continue
            other = ellipses[:idx] + ellipses[idx + 1:]
            if other and bool(is_double_ellipse_material(np.array([xs]), np.array([ys]), other)[0]):
                continue
            n_side += nz
    return (n_plane + n_top + n_side, {'plane': n_plane, 'top': n_top, 'side': n_side})


In [5]:
def is_double_ellipse_material(x: np.ndarray, y: np.ndarray, ellipses: list[dict[str, float]]) -> np.ndarray:
    inside = np.zeros(x.shape, dtype=bool)
    for ellipse in ellipses:
        dx = x - ellipse['cx']
        dy = y - ellipse['cy']
        ca = np.cos(ellipse['angle'])
        sa = np.sin(ellipse['angle'])
        x_local = ca * dx + sa * dy
        y_local = -sa * dx + ca * dy
        inside |= (x_local / ellipse['a']) ** 2 + (y_local / ellipse['b']) ** 2 <= 1.0
    return inside


In [6]:
def ellipse_perimeter(a: float, b: float) -> float:
    h = ((a - b) / (a + b)) ** 2
    return float(np.pi * (a + b) * (1.0 + 3.0 * h / (10.0 + np.sqrt(4.0 - 3.0 * h))))


In [8]:
args = SETTINGS

args.out_dir.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(args.random_seed)

n_sites, site_counts = count_binding_sites(args.lx_nm, args.ly_nm, args.protein_diameter_nm, args.spacing_factor, args.height_nm, args.ellipse_width_nm, args.ellipse_length_nm, args.ellipse_centre_spacing_nm, args.ellipse_tilt_deg)

times = np.arange(0, args.time_stop_s + args.time_step_s, args.time_step_s, dtype=np.int32)

thickness_file = args.out_dir / f'protein_total_thickness_0_to_{args.time_stop_s:04d}s.csv'

metadata_file = args.out_dir / 'protein_langmuir_records_metadata.csv'

trace_columns = [f'thicknessNm_t{int(t):04d}s' for t in times]

header = ['recordId', 'concentrationIndex', 'monteCarloIndex', 'concentrationM', *trace_columns]

metadata_header = ['recordId', 'concentrationIndex', 'monteCarloIndex', 'concentrationM', 'finalThicknessNm', 'finalOccupiedSites']

pd.DataFrame(columns=header).to_csv(thickness_file, index=False)

pd.DataFrame(columns=metadata_header).to_csv(metadata_file, index=False)

print(n_sites, site_counts)

5852 {'plane': 2656, 'top': 736, 'side': 2460}


In [15]:
record_id = np.arange(1, args.n_records + 1, dtype=np.int64)
n = args.n_records
concentration_min_m = args.concentration_min_m
concentration_max_m = args.concentration_max_m
log_c = np.log10(concentration_min_m) + (np.log10(concentration_max_m) - np.log10(concentration_min_m)) * rng.random(n)
concentration = np.power(10.0, log_c)

dt = args.time_step_s
kon = args.kon
koff = args.koff
protein_diameter_nm = args.protein_diameter_nm

p_on = np.clip(kon * concentration * dt, 0.0, 1.0)
p_off = float(np.clip(koff * dt, 0.0, 1.0))

occupied = np.zeros(len(concentration), dtype=np.int32)
thickness = np.zeros((len(concentration), len(times)), dtype=np.float32)
scale = np.pi * protein_diameter_nm / 6.0
for t_idx in range(1, len(times)):
    empty = n_sites - occupied
    n_bind = rng.binomial(empty, p_on).astype(np.int32)
    n_unbind = rng.binomial(occupied, p_off).astype(np.int32)
    occupied = np.clip(occupied + n_bind - n_unbind, 0, n_sites).astype(np.int32)
    thickness[:, t_idx] = scale * (occupied / n_sites)
thickness, occupied = (thickness, occupied)
base = pd.DataFrame({'recordId': record_id, 'concentrationIndex': record_id, 'monteCarloIndex': np.ones(args.n_records, dtype=np.int16), 'concentrationM': concentration})
pd.concat([base, pd.DataFrame(thickness, columns=trace_columns)], axis=1).to_csv(thickness_file, mode='a', header=False, index=False)
metadata = base.copy()
metadata['finalThicknessNm'] = thickness[:, -1]
metadata['finalOccupiedSites'] = occupied
metadata.to_csv(metadata_file, mode='a', header=False, index=False)
print(f'Wrote {args.n_records} traces')


Wrote 10000 traces


In [16]:
summary = pd.DataFrame([{'nRecords': args.n_records, 'nConcentrations': args.n_records, 'nMonteCarlo': 1, 'concentrationMinM': args.concentration_min_m, 'concentrationMaxM': args.concentration_max_m, 'concentrationSampling': 'loguniform', 'randomSeed': args.random_seed, 'dtSeconds': args.time_step_s, 'kon_Minv_sinv': args.kon, 'koff_sinv': args.koff, 'periodXNm': args.lx_nm, 'periodYNm': args.ly_nm, 'heightNm': args.height_nm, 'widthNm': args.ellipse_width_nm, 'lengthNm': args.ellipse_length_nm, 'centreSpacingNm': args.ellipse_centre_spacing_nm, 'thetaDeg': args.ellipse_tilt_deg, 'nBindingSites': n_sites, 'nPlaneSites': site_counts['plane'], 'nTopSites': site_counts['top'], 'nSideSites': site_counts['side'], 'proteinDiameterNm': args.protein_diameter_nm, 'thicknessMapping': 'coverage*pi*D/6', 'saturationThicknessNm': np.pi * args.protein_diameter_nm / 6.0, 'bindingSimulationMode': 'count'}])

summary.to_csv(args.out_dir / 'protein_langmuir_dataset_summary.csv', index=False)

(args.out_dir / 'fixed_rate_langmuir_dataset_summary.json').write_text(json.dumps({**summary.iloc[0].to_dict(), 'timeStopS': args.time_stop_s}, indent=2), encoding='utf-8')

print(summary.to_string(index=False))


 nRecords  nConcentrations  nMonteCarlo  concentrationMinM  concentrationMaxM concentrationSampling  randomSeed  dtSeconds  kon_Minv_sinv  koff_sinv  periodXNm  periodYNm  heightNm  widthNm  lengthNm  centreSpacingNm  thetaDeg  nBindingSites  nPlaneSites  nTopSites  nSideSites  proteinDiameterNm thicknessMapping  saturationThicknessNm bindingSimulationMode
    10000            10000            1       3.162278e-11       1.000000e-08            loguniform           7         60        55800.0   0.000018      421.5     345.63     100.0   236.04      84.3            185.5      69.5           5852         2656        736        2460               6.52  coverage*pi*D/6               3.413864                 count
